# Bias in Bios Concept-QA

Train the reusable Concept-QA module for `LabHC/bias_in_bios`.

Main choices:

- input encoder: `sentence-transformers/all-MiniLM-L6-v2`
- query vocabulary: fixed BIOS concept set
- supervision: calibrated description-similarity soft targets
- artifact: small Concept-QA checkpoint and optional split-level cache

In [ ]:
# %pip install -e ..
# %pip install datasets sentence-transformers scikit-learn -q

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from staq.config import default_paths, detect_device
from staq.models import ConceptNet2
from staq.training import seed_everything

In [2]:
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
paths = default_paths(repo_root)
paths.ensure_artifact_dirs()

device = detect_device()
random_seed = 13
seed_everything(random_seed)

encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384

dataset_name = "LabHC/bias_in_bios"
text_column = "hard_text"
target_column = "profession"
sensitive_column = "gender"

profession_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = ["male", "female"]

{
    "device": str(device),
    "encoder_name": encoder_name,
    "embedding_dim": embedding_dim,
    "dataset_name": dataset_name,
}

{'device': 'cuda',
 'encoder_name': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_dim': 384,
 'dataset_name': 'LabHC/bias_in_bios'}

In [3]:
raw_train = load_dataset(dataset_name, split="train")
raw_dev = load_dataset(dataset_name, split="dev")
raw_test = load_dataset(dataset_name, split="test")

{
    "train": len(raw_train),
    "dev": len(raw_dev),
    "test": len(raw_test),
    "columns": raw_train.column_names,
}

{'train': 257478,
 'dev': 39642,
 'test': 99069,
 'columns': ['hard_text', 'profession', 'gender']}

In [4]:
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
concepts_df = pd.read_csv(concepts_path)

concept_names = concepts_df["concept"].tolist()
concept_texts = concepts_df["description"].tolist()
sensitive_concepts = concepts_df.query("kind != 'utility'")["concept"].tolist()
sensitive_mask = torch.tensor(concepts_df["kind"].ne("utility").to_numpy(), dtype=torch.bool)

{
    "num_concepts": len(concepts_df),
    "by_kind": concepts_df["kind"].value_counts().to_dict(),
    "num_sensitive_concepts": int(sensitive_mask.sum()),
}

{'num_concepts': 40,
 'by_kind': {'utility': 33, 'proxy_sensitive': 5, 'direct_sensitive': 2},
 'num_sensitive_concepts': 7}

In [5]:
encoder = SentenceTransformer(encoder_name, device=str(device))

concept_embeddings = encoder.encode(
    concept_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

concept_dictionary = concept_embeddings.T.contiguous()

{
    "concept_embeddings": tuple(concept_embeddings.shape),
    "concept_dictionary": tuple(concept_dictionary.shape),
}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'concept_embeddings': (40, 384), 'concept_dictionary': (384, 40)}

In [6]:
sample_rows = raw_train.select(range(8))
sample_texts = [row[text_column] for row in sample_rows]

sample_embeddings = encoder.encode(
    sample_texts,
    batch_size=8,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).to(device)

sample_concept_qa_inputs = torch.cat(
    [
        sample_embeddings.unsqueeze(1).repeat(1, len(concept_names), 1),
        concept_embeddings.unsqueeze(0).repeat(len(sample_texts), 1, 1),
    ],
    dim=2,
).reshape(len(sample_texts) * len(concept_names), -1)

{
    "sample_embeddings": tuple(sample_embeddings.shape),
    "concept_qa_input_shape": tuple(sample_concept_qa_inputs.shape),
    "expected_concept_qa_input_dim": embedding_dim * 2,
}

{'sample_embeddings': (8, 384),
 'concept_qa_input_shape': (320, 768),
 'expected_concept_qa_input_dim': 768}

In [7]:
pd.DataFrame(
    {
        "profession": [profession_names[row[target_column]] for row in sample_rows],
        "gender": [gender_names[row[sensitive_column]] for row in sample_rows],
        "text": sample_texts,
    }
)

,profession,gender,text
0,professor,male,He is also the project lead of and major contr...
1,nurse,female,"She is able to assess, diagnose and treat mino..."
2,attorney,female,"Prior to law school, Brittni graduated magna c..."
3,journalist,male,He regularly contributes to India’s First Onli...
4,professor,male,He completed his medical degree at Northwester...
5,attorney,male,Steve has practiced law in Kentucky for over 3...
6,professor,male,"His research is in information archiving, anal..."
7,poet,female,"Trained as a doctor, she lives in Algiers wher..."


## Qwen Teacher Probe

Small yes/no LLM check before generating full Concept-QA targets.

In [9]:
qwen_teacher_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_probe_size = 1
qwen_batch_size = 4
qwen_max_length = 1024
qwen_max_new_tokens = 4

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_teacher_name)
qwen_tokenizer.padding_side = "left"
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model_kwargs = {}
if device.type == "cuda":
    qwen_model_kwargs["torch_dtype"] = torch.float16

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_teacher_name,
    **qwen_model_kwargs,
).to(device)
qwen_model.eval()
qwen_model.generation_config.pad_token_id = qwen_tokenizer.pad_token_id


def build_qwen_teacher_prompt(description, text):
    description = description.strip()
    if description.lower().startswith("this biography "):
        concept_question = f"Does the biography {description[len('this biography '):]}?"
    else:
        concept_question = f"Does the biography explicitly satisfy this concept: {description}?"

    messages = [
        {
            "role": "system",
            "content": "You answer concept-presence questions about biographies. Answer only Yes or No.",
        },
        {
            "role": "user",
            "content": (
                f"Question: {concept_question}\n"
                f"Biography: `{text}`"
            ),
        },
    ]
    return qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def parse_yes_no(answer):
    normalized = answer.strip().lower()
    if normalized.startswith("yes"):
        return 1.0
    if normalized.startswith("no"):
        return 0.0
    return float("nan")


qwen_prompt_rows = []
for row_index, text in enumerate(sample_texts[:qwen_probe_size]):
    for concept_index, description in enumerate(concept_texts):
        qwen_prompt_rows.append(
            {
                "row": row_index,
                "concept_index": concept_index,
                "prompt": build_qwen_teacher_prompt(description, text),
            }
        )

qwen_outputs = []
with torch.no_grad():
    for start in range(0, len(qwen_prompt_rows), qwen_batch_size):
        batch_rows = qwen_prompt_rows[start : start + qwen_batch_size]
        encoded = qwen_tokenizer(
            [row["prompt"] for row in batch_rows],
            padding=True,
            truncation=True,
            max_length=qwen_max_length,
            return_tensors="pt",
        ).to(device)
        generated = qwen_model.generate(
            **encoded,
            max_new_tokens=qwen_max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.pad_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
        )
        generated_answers = qwen_tokenizer.batch_decode(
            generated[:, encoded["input_ids"].shape[1] :],
            skip_special_tokens=True,
        )
        for row, answer in zip(batch_rows, generated_answers):
            qwen_outputs.append(
                {
                    "row": row["row"],
                    "concept_index": row["concept_index"],
                    "answer": answer.strip(),
                    "qwen_yes_score": parse_yes_no(answer),
                }
            )

qwen_probe = pd.DataFrame(qwen_outputs)
qwen_yes_scores = torch.full((qwen_probe_size, len(concept_names)), float("nan"), dtype=torch.float32)
for output in qwen_outputs:
    qwen_yes_scores[output["row"], output["concept_index"]] = output["qwen_yes_score"]

bi_encoder_scores = (sample_embeddings[:qwen_probe_size] @ concept_embeddings.T).detach().float().cpu()


def show_qwen_teacher_concepts(row_index=0):
    row_outputs = qwen_probe[qwen_probe["row"].eq(row_index)].copy()
    row_outputs["concept"] = [concept_names[idx] for idx in row_outputs["concept_index"]]
    row_outputs["kind"] = [concepts_df.loc[idx, "kind"] for idx in row_outputs["concept_index"]]
    row_outputs["description"] = [concept_texts[idx] for idx in row_outputs["concept_index"]]
    row_outputs["bi_encoder_similarity"] = [
        float(bi_encoder_scores[row_index, idx]) for idx in row_outputs["concept_index"]
    ]
    print(f"profession: {profession_names[sample_rows[row_index][target_column]]}")
    print(f"gender: {gender_names[sample_rows[row_index][sensitive_column]]}")
    print(sample_texts[row_index])
    return row_outputs.sort_values(
        ["qwen_yes_score", "concept"], ascending=[False, True]
    )[
        ["concept", "kind", "answer", "qwen_yes_score", "bi_encoder_similarity", "description"]
    ]


show_qwen_teacher_concepts(0)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

profession: professor
gender: male
He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates


,concept,kind,answer,qwen_yes_score,bi_encoder_similarity,description
13,advanced_degree_or_training,utility,Yes,1.0,0.246945,"Biography mentions graduate degrees, doctoral ..."
12,research_publications,utility,Yes,1.0,0.196503,"Biography mentions research, publications, gra..."
14,software_development,utility,Yes,1.0,0.438074,"Biography mentions writing software, programmi..."
11,university_affiliation,utility,Yes,1.0,0.237799,"Biography mentions universities, departments, ..."
38,appearance_or_body_presentation,proxy_sensitive,No,0.0,0.117060,"Biography emphasizes appearance, beauty, body ..."
16,building_or_spatial_design,utility,No,0.0,0.282557,"Biography mentions buildings, architecture, sp..."
31,business_or_client_service,utility,No,0.0,0.164809,"Biography mentions clients, consulting, busine..."
19,camera_or_photography_work,utility,No,0.0,0.199328,"Biography mentions photography, cameras, portr..."
10,classroom_instruction,utility,No,0.0,0.213470,"Biography mentions classroom teaching, pupils,..."
2,clinical_care,utility,No,0.0,0.082183,"Biography mentions direct clinical care, diagn..."


In [10]:
print(concepts_df.loc[concepts_df["concept"].eq("direct_gender_pronouns"), "description"].iloc[0])
print(concepts_df[["concept", "description"]].tail(8))

Biography uses gendered personal pronouns.
                             concept  \
32      financial_or_accounting_work   
33            direct_gender_pronouns   
34                   gendered_titles   
35             gendered_family_roles   
36  parenting_or_caregiving_mentions   
37              gendered_name_signal   
38   appearance_or_body_presentation   
39    gendered_organization_or_award   

                                          description  
32  Biography mentions accounting, taxes, payroll,...  
33         Biography uses gendered personal pronouns.  
34  Biography uses explicitly gendered titles or h...  
35  Biography mentions family roles that often rev...  
36  Biography mentions parenting, children, caregi...  
37  Biography contains first-name information that...  
38  Biography emphasizes appearance, beauty, body ...  
39  Biography mentions gendered organizations, awa...  


In [33]:
sensitive_indices = concepts_df.index[concepts_df["kind"].ne("utility")].to_numpy()

sensitive_probe_rows = []
for row_index in range(qwen_probe_size):
    row_answers = qwen_probe[qwen_probe["row"].eq(row_index)].set_index("concept_index")
    for concept_index in sensitive_indices:
        answer = row_answers.loc[concept_index, "answer"]
        sensitive_probe_rows.append(
            {
                "row": row_index,
                "profession": profession_names[sample_rows[row_index][target_column]],
                "gender": gender_names[sample_rows[row_index][sensitive_column]],
                "concept": concept_names[concept_index],
                "kind": concepts_df.loc[concept_index, "kind"],
                "answer": answer,
                "qwen_yes_score": float(qwen_yes_scores[row_index, concept_index]),
                "bi_encoder_similarity": float(bi_encoder_scores[row_index, concept_index]),
                "text": sample_texts[row_index],
            }
        )

sensitive_probe = pd.DataFrame(sensitive_probe_rows)
sensitive_score_matrix = sensitive_probe.pivot_table(
    index=["row", "profession", "gender"],
    columns="concept",
    values="qwen_yes_score",
).reset_index()

positive_sensitive_scores = sensitive_probe[
    sensitive_probe["qwen_yes_score"].eq(1.0)
].sort_values(["row", "concept"])

display(sensitive_score_matrix)
display(positive_sensitive_scores[["row", "profession", "gender", "concept", "kind", "answer", "text"]])

concept,row,profession,gender,appearance_or_body_presentation,direct_gender_pronouns,gendered_family_roles,gendered_organization_or_award,gendered_titles,parenting_or_caregiving_mentions
0,0,professor,male,0.444092,0.959473,0.727051,0.317627,0.795410,0.186401
1,1,nurse,female,0.457520,0.968262,0.903320,0.609863,0.878906,0.279541
2,2,attorney,female,0.202026,0.989258,0.984863,0.963379,0.970703,0.269287
3,3,journalist,male,0.069885,0.979004,0.971680,0.476074,0.881836,0.018051
4,4,professor,male,0.451416,0.981934,0.729492,0.911133,0.951660,0.119263
5,5,attorney,male,0.021439,0.978027,0.918457,0.898438,0.973633,0.001323
6,6,professor,male,0.062317,0.963867,0.890625,0.839355,0.976562,0.000975
7,7,poet,female,0.090149,0.967773,0.964355,0.799805,0.937500,0.019135


,row,profession,gender,concept,kind,nli_entailment,text
0,0,professor,male,direct_gender_pronouns,direct_sensitive,0.959473,He is also the project lead of and major contr...
1,0,professor,male,gendered_titles,direct_sensitive,0.795410,He is also the project lead of and major contr...
2,0,professor,male,gendered_family_roles,proxy_sensitive,0.727051,He is also the project lead of and major contr...
6,1,nurse,female,direct_gender_pronouns,direct_sensitive,0.968262,"She is able to assess, diagnose and treat mino..."
8,1,nurse,female,gendered_family_roles,proxy_sensitive,0.903320,"She is able to assess, diagnose and treat mino..."
7,1,nurse,female,gendered_titles,direct_sensitive,0.878906,"She is able to assess, diagnose and treat mino..."
11,1,nurse,female,gendered_organization_or_award,proxy_sensitive,0.609863,"She is able to assess, diagnose and treat mino..."
12,2,attorney,female,direct_gender_pronouns,direct_sensitive,0.989258,"Prior to law school, Brittni graduated magna c..."
14,2,attorney,female,gendered_family_roles,proxy_sensitive,0.984863,"Prior to law school, Brittni graduated magna c..."
13,2,attorney,female,gendered_titles,direct_sensitive,0.970703,"Prior to law school, Brittni graduated magna c..."


In [ ]:
encoding_batch_size = 256
raw_splits = {
    "train": raw_train,
    "dev": raw_dev,
    "test": raw_test,
}


def encode_bios_split(split):
    texts = [row[text_column] for row in split]
    embeddings = encoder.encode(
        texts,
        batch_size=encoding_batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).to(device)
    professions = [profession_names[row[target_column]] for row in split]
    genders = [gender_names[row[sensitive_column]] for row in split]
    return {
        "texts": texts,
        "embeddings": embeddings,
        "similarity_scores": embeddings @ concept_embeddings.T,
        "profession_labels": professions,
        "gender_labels": genders,
        "profession_ids": torch.tensor([row[target_column] for row in split], dtype=torch.long),
        "gender_ids": torch.tensor([row[sensitive_column] for row in split], dtype=torch.long),
    }


encoded_splits = {
    split_name: encode_bios_split(split)
    for split_name, split in raw_splits.items()
}

{
    split_name: {
        "embeddings": tuple(split_data["embeddings"].shape),
        "similarity_scores": tuple(split_data["similarity_scores"].shape),
    }
    for split_name, split_data in encoded_splits.items()
}

In [ ]:
train_similarity_scores = encoded_splits["train"]["similarity_scores"]
concept_medians = train_similarity_scores.median(dim=0).values
concept_stds = train_similarity_scores.std(dim=0).clamp_min(1e-6)
calibration_temperature = 1.5


def calibrate_similarity_scores(similarity_scores):
    return torch.sigmoid(
        (similarity_scores - concept_medians.unsqueeze(0))
        / (calibration_temperature * concept_stds.unsqueeze(0))
    )


for split_data in encoded_splits.values():
    split_data["calibrated_soft_targets"] = calibrate_similarity_scores(split_data["similarity_scores"])

train_targets = encoded_splits["train"]["calibrated_soft_targets"]
calibrated_summary = pd.DataFrame(
    {
        "concept": concept_names,
        "kind": concepts_df["kind"],
        "mean": train_targets.mean(dim=0).cpu().numpy(),
        "std": train_targets.std(dim=0).cpu().numpy(),
        "p05": torch.quantile(train_targets, 0.05, dim=0).cpu().numpy(),
        "p50": torch.quantile(train_targets, 0.50, dim=0).cpu().numpy(),
        "p95": torch.quantile(train_targets, 0.95, dim=0).cpu().numpy(),
    }
)

calibrated_summary.sort_values("mean", ascending=False)

In [ ]:
def show_top_calibrated_concepts(row_index, split_name="train", top_k=8):
    split_data = encoded_splits[split_name]
    scores = split_data["calibrated_soft_targets"][row_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    print(f"profession: {split_data['profession_labels'][row_index]}")
    print(f"gender: {split_data['gender_labels'][row_index]}")
    print(split_data["texts"][row_index])
    return pd.DataFrame(
        {
            "concept": [concept_names[idx] for idx in top_indices],
            "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
            "score": scores[top_indices],
        }
    )


show_top_calibrated_concepts(0)

In [ ]:
def show_high_calibrated_examples(concept, split_name="train", top_k=5):
    split_data = encoded_splits[split_name]
    concept_index = concept_names.index(concept)
    scores = split_data["calibrated_soft_targets"][:, concept_index].detach().cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame(
        {
            "score": scores[top_indices],
            "profession": [split_data["profession_labels"][idx] for idx in top_indices],
            "gender": [split_data["gender_labels"][idx] for idx in top_indices],
            "text": [split_data["texts"][idx] for idx in top_indices],
        }
    )


show_high_calibrated_examples("software_development")

In [ ]:
batch_size = 256

def make_concept_qa_dataset(split_name):
    split_data = encoded_splits[split_name]
    return TensorDataset(
        split_data["embeddings"].cpu(),
        split_data["calibrated_soft_targets"].cpu(),
    )


train_dataset = make_concept_qa_dataset("train")
dev_dataset = make_concept_qa_dataset("dev")
test_dataset = make_concept_qa_dataset("test")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

{
    "train_examples": len(train_dataset),
    "dev_examples": len(dev_dataset),
    "test_examples": len(test_dataset),
    "num_concepts": len(concept_names),
}

In [ ]:
def build_text_concept_qa_inputs(text_embeddings, concept_embeddings):
    batch_size = text_embeddings.size(0)
    num_concepts = concept_embeddings.size(0)
    text_rep = text_embeddings.unsqueeze(1).repeat(1, num_concepts, 1)
    concept_rep = concept_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)
    return torch.cat([text_rep, concept_rep], dim=2).reshape(batch_size * num_concepts, -1)


def tensor_corrcoef(left, right):
    left = left.flatten()
    right = right.flatten()
    left = left - left.mean()
    right = right - right.mean()
    denom = left.norm() * right.norm()
    if float(denom) == 0.0:
        return torch.tensor(float("nan"), device=left.device)
    return (left @ right) / denom


def rank_tensor(values):
    order = values.flatten().argsort()
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(len(order), device=values.device, dtype=torch.float32)
    return ranks.view_as(values)


@torch.no_grad()
def evaluate_concept_qa_soft(model, loader):
    model.eval()
    losses = []
    hard_accuracies = []
    pred_batches = []
    target_batches = []

    for text_batch, target_batch in loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view_as(target_batch)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)
        preds = (probs >= 0.5).float()
        hard_targets = (target_batch >= 0.5).float()

        losses.append(float(loss.item()))
        hard_accuracies.append(float((preds == hard_targets).float().mean().item()))
        pred_batches.append(probs.detach().cpu())
        target_batches.append(target_batch.detach().cpu())

    all_preds = torch.cat(pred_batches)
    all_targets = torch.cat(target_batches)
    pearson = tensor_corrcoef(all_preds, all_targets)
    spearman = tensor_corrcoef(rank_tensor(all_preds), rank_tensor(all_targets))

    return {
        "loss": float(np.mean(losses)),
        "mae": float((all_preds - all_targets).abs().mean().item()),
        "rmse": float(torch.sqrt(((all_preds - all_targets) ** 2).mean()).item()),
        "pearson": float(pearson.item()),
        "spearman": float(spearman.item()),
        "hard_accuracy": float(np.mean(hard_accuracies)),
    }

In [ ]:
concept_qa_model = ConceptNet2(embed_dims=embedding_dim).to(device)
optimizer = torch.optim.AdamW(concept_qa_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 5

concept_qa_history = []
for epoch in range(1, num_epochs + 1):
    concept_qa_model.train()
    train_losses = []

    for text_batch, target_batch in train_loader:
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)

        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = concept_qa_model(inputs).view_as(target_batch)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    dev_metrics = evaluate_concept_qa_soft(concept_qa_model, dev_loader)
    concept_qa_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "dev_loss": dev_metrics["loss"],
            "dev_mae": dev_metrics["mae"],
            "dev_rmse": dev_metrics["rmse"],
            "dev_pearson": dev_metrics["pearson"],
            "dev_spearman": dev_metrics["spearman"],
            "dev_hard_accuracy": dev_metrics["hard_accuracy"],
        }
    )

concept_qa_test_metrics = evaluate_concept_qa_soft(concept_qa_model, test_loader)

pd.DataFrame(concept_qa_history), concept_qa_test_metrics

In [ ]:
@torch.no_grad()
def predict_concept_scores_for_row(row_index, split_name="dev"):
    concept_qa_model.eval()
    split_data = encoded_splits[split_name]
    text_embedding = split_data["embeddings"][row_index : row_index + 1].to(device)
    inputs = build_text_concept_qa_inputs(text_embedding, concept_embeddings)
    logits = concept_qa_model(inputs).view(1, len(concept_names))
    return torch.sigmoid(logits).squeeze(0).cpu().numpy()


row_index = 0
split_name = "dev"
model_scores = predict_concept_scores_for_row(row_index, split_name=split_name)
target_scores = encoded_splits[split_name]["calibrated_soft_targets"][row_index].cpu().numpy()
top_indices = np.argsort(model_scores)[::-1][:8]

pd.DataFrame(
    {
        "concept": [concept_names[idx] for idx in top_indices],
        "kind": [concepts_df.loc[idx, "kind"] for idx in top_indices],
        "model_score": model_scores[top_indices],
        "target_score": target_scores[top_indices],
    }
)

In [ ]:
qa_experiment_name = "bias_in_bios_concept_qa_prompt_full"
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"
qa_cache_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_cache.pt"
save_split_cache = False

target_metadata = {
    "target_strategy": "calibrated_description_similarity_soft_targets",
    "concept_description_column": "description",
}

checkpoint_payload = {
    "model_state_dict": concept_qa_model.state_dict(),
    "embedding_dim": embedding_dim,
    "encoder_name": encoder_name,
    "concepts": concepts_df.to_dict(orient="records"),
    "concepts_path": str(concepts_path.relative_to(repo_root)),
    "concept_embeddings": concept_embeddings.detach().cpu(),
    "concept_dictionary": concept_dictionary.detach().cpu(),
    "calibration_medians": concept_medians.detach().cpu(),
    "calibration_stds": concept_stds.detach().cpu(),
    "calibration_temperature": calibration_temperature,
    "target_metadata": target_metadata,
    "test_metrics": concept_qa_test_metrics,
}

torch.save(checkpoint_payload, qa_checkpoint)

if save_split_cache:
    cache_payload = {
        "encoder_name": encoder_name,
        "embedding_dim": embedding_dim,
        "concepts": concepts_df.to_dict(orient="records"),
        "concept_embeddings": concept_embeddings.detach().cpu(),
        "concept_dictionary": concept_dictionary.detach().cpu(),
        "calibration_medians": concept_medians.detach().cpu(),
        "calibration_stds": concept_stds.detach().cpu(),
        "calibration_temperature": calibration_temperature,
        "target_metadata": target_metadata,
        "splits": {
            split_name: {
                "embeddings": split_data["embeddings"].detach().cpu(),
                "similarity_scores": split_data["similarity_scores"].detach().cpu(),
                "calibrated_soft_targets": split_data["calibrated_soft_targets"].detach().cpu(),
                "profession_ids": split_data["profession_ids"].cpu(),
                "gender_ids": split_data["gender_ids"].cpu(),
                "profession_labels": split_data["profession_labels"],
                "gender_labels": split_data["gender_labels"],
            }
            for split_name, split_data in encoded_splits.items()
        },
    }
    torch.save(cache_payload, qa_cache_path)

history_payload = {
    "history": concept_qa_history,
    "test_metrics": concept_qa_test_metrics,
    "target_metadata": target_metadata,
}
with open(qa_history_path, "w", encoding="utf-8") as handle:
    json.dump(history_payload, handle, indent=2)

{
    "checkpoint": str(qa_checkpoint.relative_to(repo_root)),
    "history": str(qa_history_path.relative_to(repo_root)),
    "cache": str(qa_cache_path.relative_to(repo_root)) if save_split_cache else None,
}